<a href="https://colab.research.google.com/github/silvia-dev-prog/ai-agents-for-beginners/blob/main/C%C3%B3pia_de_RL_Aula1_1_Ambiente_Controle_de_Temperatura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from scipy.integrate import solve_ivp

In [2]:
# Parâmetros do reator
params = {
    "Cp": 4.18e3,  # Capacidade térmica do reator (J/kg·K)
    "V": 10.0,  # Volume do reator (m³)
    "delta_Hr": -2e6,  # Calor de reação (J/mol)
    "k": 10,  # Constante de velocidade (1/s)
    "Ea": 8.314 * 4000,  # Energia de ativação (J/mol)
    "R": 8.314,  # Constante universal dos gases (J/mol·K)
    "T_cool": 300.0,  # Temperatura do fluido resfriante (K)
    "Cp_cool": 4.18e3,  # Capacidade térmica do fluido resfriante (J/kg·K)
    "T_setpoint": 350.0,  # Setpoint de temperatura (K)
}

In [3]:
# Estado inicial
initial_state = {
    "T": 300.0,  # Temperatura inicial (K)
}

In [4]:
# Modelo do reator
def reactor_model(t, y, mdot, params):
    T, CA = y
    Q_reaction = (
        -params["delta_Hr"]
        * params["k"]
        * CA
        * np.exp(-params["Ea"] / (params["R"] * T))
    )*params["V"]
    Q_cooling = mdot * params["Cp_cool"] * (T - params["T_cool"])
    #print(Q_reaction , Q_cooling)
    dTdt = (Q_reaction - Q_cooling) / (params["Cp"] * params["V"])
    dCAdt = -params["k"] * CA * np.exp(-params["Ea"] / (params["R"] * T))
    return [dTdt, dCAdt]

In [5]:
# Função para executar um passo do ambiente
def step(state, action, params, mdot, CA):
    T = state["T"]
    # Atualiza a vazão com base na ação
    mdot = mdot + action
    mdot = max(mdot, 0)  # Garante que a vazão seja positiva

    # Simula o próximo estado
    t_span = [0, 10]  # Passo de tempo fixo de 10 segundo
    y0 = [T, CA]  # Condições iniciais atualizadas
    sol = solve_ivp(
        reactor_model, t_span, y0, args=(mdot, params), method="RK45"
    )

    # Atualiza os estados
    T, CA = sol.y[:, -1]  # Atualiza temperatura e concentração

    # Calcula a recompensa
    error = abs(T - params["T_setpoint"]) / params["T_setpoint"]
    reward = 1 - error

    # Retorna o novo estado, a recompensa, o valor atualizado de mdot e CA
    new_state = {"T": T}
    return new_state, reward, False, mdot, CA

In [6]:
# Função para reiniciar o ambiente
def reset(params):
    return initial_state.copy(), 0, 10  # Retorna estado inicial, vazão e concentração iniciais

In [7]:
step(state = {"T": 300},
     action = 0,
     params=params,
     mdot=2,
     CA=10)

({'T': np.float64(300.33886570235603)},
 np.float64(0.8581110448638744),
 False,
 2,
 np.float64(9.998364491133415))

In [8]:
state, mdot, CA = reset(params)
print(state, mdot, CA)

{'T': 300.0} 0 10


In [9]:
action = 0
mdot=0
state, reward, done, mdot, CA = step(state, action, params, mdot, CA)
print(state, reward, done, mdot, CA)

{'T': np.float64(300.7884958148547)} 0.8593957023281562 False 0 9.998352043746953


In [10]:
# Inicializa o ambiente
state, mdot, CA = reset(params)

# Executa passos de simulação
for step_idx in range(40):
    action = 0.001
    state, reward, done, mdot, CA = step(state, action, params, mdot, CA)
    print(f"Step {step_idx + 1}: State={{'T': {state['T']:.2f}}}, mdot={mdot:.2f}, CA={CA:.2f}, Reward={reward:.2f}")


Step 1: State={'T': 300.79}, mdot=0.00, CA=10.00, Reward=0.86
Step 2: State={'T': 301.60}, mdot=0.00, CA=10.00, Reward=0.86
Step 3: State={'T': 302.44}, mdot=0.00, CA=9.99, Reward=0.86
Step 4: State={'T': 303.31}, mdot=0.00, CA=9.99, Reward=0.87
Step 5: State={'T': 304.21}, mdot=0.01, CA=9.99, Reward=0.87
Step 6: State={'T': 305.13}, mdot=0.01, CA=9.99, Reward=0.87
Step 7: State={'T': 306.08}, mdot=0.01, CA=9.99, Reward=0.87
Step 8: State={'T': 307.05}, mdot=0.01, CA=9.98, Reward=0.88
Step 9: State={'T': 308.06}, mdot=0.01, CA=9.98, Reward=0.88
Step 10: State={'T': 309.09}, mdot=0.01, CA=9.98, Reward=0.88
Step 11: State={'T': 310.16}, mdot=0.01, CA=9.98, Reward=0.89
Step 12: State={'T': 311.25}, mdot=0.01, CA=9.98, Reward=0.89
Step 13: State={'T': 312.38}, mdot=0.01, CA=9.97, Reward=0.89
Step 14: State={'T': 313.54}, mdot=0.01, CA=9.97, Reward=0.90
Step 15: State={'T': 314.74}, mdot=0.02, CA=9.97, Reward=0.90
Step 16: State={'T': 315.97}, mdot=0.02, CA=9.96, Reward=0.90
Step 17: State=

In [11]:
# Inicializa o ambiente
state, mdot, CA = reset(params)

# Executa passos de simulação
for step_idx in range(20):
    action = +0.1#np.random.choice([-0.1, 0, +0.1])  # Ações incrementais
    state, reward, done, mdot, CA = step(state, action, params, mdot, CA)
    print(f"Step {step_idx + 1}: State={{'T': {state['T']:.2f}}}, mdot={mdot:.2f}, CA={CA:.2f}, Reward={reward:.2f}")


Step 1: State={'T': 300.75}, mdot=0.10, CA=10.00, Reward=0.86
Step 2: State={'T': 301.35}, mdot=0.20, CA=10.00, Reward=0.86
Step 3: State={'T': 301.72}, mdot=0.30, CA=9.99, Reward=0.86
Step 4: State={'T': 301.84}, mdot=0.40, CA=9.99, Reward=0.86
Step 5: State={'T': 301.78}, mdot=0.50, CA=9.99, Reward=0.86
Step 6: State={'T': 301.60}, mdot=0.60, CA=9.99, Reward=0.86
Step 7: State={'T': 301.39}, mdot=0.70, CA=9.99, Reward=0.86
Step 8: State={'T': 301.19}, mdot=0.80, CA=9.99, Reward=0.86
Step 9: State={'T': 301.02}, mdot=0.90, CA=9.98, Reward=0.86
Step 10: State={'T': 300.88}, mdot=1.00, CA=9.98, Reward=0.86
Step 11: State={'T': 300.78}, mdot=1.10, CA=9.98, Reward=0.86
Step 12: State={'T': 300.70}, mdot=1.20, CA=9.98, Reward=0.86
Step 13: State={'T': 300.64}, mdot=1.30, CA=9.98, Reward=0.86
Step 14: State={'T': 300.58}, mdot=1.40, CA=9.98, Reward=0.86
Step 15: State={'T': 300.54}, mdot=1.50, CA=9.97, Reward=0.86
Step 16: State={'T': 300.50}, mdot=1.60, CA=9.97, Reward=0.86
Step 17: State=

In [12]:
# Inicializa o ambiente
state, mdot, CA = reset(params)

# Executa passos de simulação
for step_idx in range(20):
    action = np.random.choice([-0.1, 0, +0.1])  # Ações incrementais
    state, reward, done, mdot, CA = step(state, action, params, mdot, CA)
    print(f"Step {step_idx + 1}: State={{'T': {state['T']:.2f}}}, mdot={mdot:.2f}, CA={CA:.2f}, Reward={reward:.2f}")


Step 1: State={'T': 300.75}, mdot=0.10, CA=10.00, Reward=0.86
Step 2: State={'T': 301.35}, mdot=0.20, CA=10.00, Reward=0.86
Step 3: State={'T': 302.02}, mdot=0.10, CA=9.99, Reward=0.86
Step 4: State={'T': 302.64}, mdot=0.10, CA=9.99, Reward=0.86
Step 5: State={'T': 302.96}, mdot=0.20, CA=9.99, Reward=0.87
Step 6: State={'T': 302.95}, mdot=0.30, CA=9.99, Reward=0.87
Step 7: State={'T': 302.70}, mdot=0.40, CA=9.99, Reward=0.86
Step 8: State={'T': 302.32}, mdot=0.50, CA=9.99, Reward=0.86
Step 9: State={'T': 302.08}, mdot=0.50, CA=9.98, Reward=0.86
Step 10: State={'T': 301.77}, mdot=0.60, CA=9.98, Reward=0.86
Step 11: State={'T': 301.48}, mdot=0.70, CA=9.98, Reward=0.86
Step 12: State={'T': 301.23}, mdot=0.80, CA=9.98, Reward=0.86
Step 13: State={'T': 301.03}, mdot=0.90, CA=9.98, Reward=0.86
Step 14: State={'T': 300.89}, mdot=1.00, CA=9.98, Reward=0.86
Step 15: State={'T': 300.83}, mdot=1.00, CA=9.97, Reward=0.86
Step 16: State={'T': 300.81}, mdot=1.00, CA=9.97, Reward=0.86
Step 17: State=